# Computer Vision — โจทย์แบบ "แบ่งส่วนภาพ" (Segmentation)

ระบุว่าพิกเซลไหนเป็นวัตถุอะไร (เช่น แยกพื้นที่บ้าน/ถนน/ต้นไม้ในภาพ)

วิธีในโน้ตบุ๊กนี้: `YOLOv8-seg` (ultralytics) ง่ายสุดสำหรับมือใหม่ ใช้คล้าย detection แต่ได้ mask


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ output เป็น "mask ต่อพิกเซล" (ระบายว่าพิกเซลไหนเป็นคลาสอะไร) metric มักเป็น `IoU`/`Dice`
ถ้าแค่ "รูปนี้คือคลาสอะไร" -> `image_classification.ipynb` (ง่ายกว่ามาก)
ถ้าแค่ "กล่องรอบวัตถุ" -> `object_detection.ipynb`

หมายเหตุมือใหม่: segmentation ต้องมี label เป็น mask/polygon (รูปแบบ YOLO-seg) ถ้าเวลาน้อยและทำ classification ได้ ให้เลือก classification

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install ultralytics pyyaml numpy

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-segmentation-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — เตรียม data.yaml (รูปแบบ YOLO-seg)

YOLO-seg ต้องการ: โฟลเดอร์รูป + ไฟล์ label `.txt` (คลาส + พิกัด polygon เป็นค่า 0-1) ต่อรูป
ถ้าข้อมูลเป็นรูปแบบอื่น (mask PNG / COCO json) ต้องแปลงเป็น YOLO-seg ก่อน
ถ้าเทรนขึ้น error `No labels found` แปลว่ายังไม่ได้แปลงข้อมูลเป็นรูปแบบ YOLO-seg

In [ ]:
import yaml, os
data_cfg = {
    "path": os.path.abspath(DATA_DIR),
    "train": "images/train",   # << แก้: โฟลเดอร์รูป train เช่น "train/images"
    "val":   "images/val",     # << แก้: โฟลเดอร์รูป val เช่น "valid/images" (ไม่มีให้แบ่งจาก train)
    "names": {0: "object"},    # << แก้: ชื่อคลาส เช่น {0:"building",1:"road"}
}
with open("data_seg.yaml","w") as f: yaml.safe_dump(data_cfg, f, allow_unicode=True)
print(open("data_seg.yaml").read())

## ขั้นที่ 4 — เทรน YOLO-seg (ง่ายสุด)

`RLE` = วิธีบีบอัด mask ให้เป็นตัวเลขสั้น ๆ ต่อ 1 วัตถุ (comp ส่วนใหญ่ใช้แบบนี้)
`IoU`/`Dice` = วัดว่าพื้นที่ที่ทายทับกับเฉลยมากแค่ไหน ยิ่งสูงยิ่งดี

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n-seg.pt")    # << แก้: รุ่นใหม่/แม่นขึ้น เช่น "yolo11s-seg.pt" (หรือ "yolov8n-seg.pt" รุ่นเก่า)
model.train(data="data_seg.yaml", epochs=50, imgsz=640, batch=16)   # << แก้: ลอง epochs=10 ก่อน; GPU เล็ก batch=8
print(model.val())

## ขั้นที่ 5 — ทำนาย test + สร้าง submission (RLE)

comp ส่วนใหญ่ใช้ RLE ด้านล่างมีฟังก์ชัน `rle_encode` มาตรฐานให้ + resize mask กลับเป็นขนาดรูปจริงก่อน encode
(ของ ultralytics คืน mask ขนาดตาม imgsz ไม่ใช่ขนาดรูปจริง ต้อง resize ก่อน ไม่งั้น RLE จะผิดตำแหน่ง)

In [ ]:
import glob, numpy as np, pandas as pd, cv2
def rle_encode(mask):   # mask 0/1 -> สตริง RLE แบบ Kaggle (นับตามคอลัมน์)
    pix = mask.flatten(order="F"); pix = np.concatenate([[0], pix, [0]])
    runs = np.where(pix[1:] != pix[:-1])[0] + 1; runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)
TEST_IMG_DIR = os.path.join(DATA_DIR, "images/test")   # << แก้: path รูป test เช่น os.path.join(DATA_DIR,"test/images")
rows = []
for p in sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*"))):
    r = model.predict(p, verbose=False)[0]
    img_id = os.path.splitext(os.path.basename(p))[0]
    H, W = r.orig_shape   # ขนาดรูปจริง
    if r.masks is None: continue
    for ci in range(len(r.masks.data)):
        m = r.masks.data[ci].cpu().numpy().astype(np.uint8)         # ขนาดตาม imgsz ของโมเดล
        m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)  # resize กลับขนาดรูปจริง
        cls = int(r.boxes.cls[ci]) if r.boxes is not None else 0
        rows.append({"id": img_id, "class": cls, "rle": rle_encode(m)})  # << แก้คอลัมน์ให้ตรง sample
sub = pd.DataFrame(rows); sub.to_csv("submission.csv", index=False)
print("บันทึก submission.csv", sub.shape); display(sub.head())

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "yolov8 segmentation"
!kaggle competitions submissions -c {COMP} | head